# 02_ex_<agreement>_<topic> — exploration and approval drafting

Use this notebook to profile source data, draft AI-assisted suggestions, and document human-reviewed decisions before anything is enforced in pipeline execution.

**You edit**
- Agreement/topic metadata
- Source target details
- Draft contract, DQ, and classification decisions

**This notebook produces**
- Profiling evidence
- Candidate DQ/classification suggestions
- Human-reviewed contract draft and approved decisions

**AI advisory boundary**
- AI suggestions are advisory and must be human-reviewed before promotion.


## Segment 1: Load shared config and imports

What this does: loads `00_env_config` and imports exploration helpers.

Functions used: `load_fabric_config`, `build_runtime_context`, `generate_metadata_profile`, `draft_dq_rules`.

Output: configured runtime context and helper availability.


In [ ]:
%run 00_env_config


In [ ]:
import json
from pathlib import Path
import fabricops_kit.data_quality_review as dq_review
from fabricops_kit.config import bootstrap_fabric_env, get_path, load_fabric_config
from fabricops_kit.runtime import validate_notebook_name, assert_notebook_name_valid, build_runtime_context
from fabricops_kit.fabric_input_output import seed_minimal_sample_source_table
from fabricops_kit.fabric_input_output import lakehouse_table_read, warehouse_read
from fabricops_kit import (
    normalize_contract_dict,
    validate_contract_dict,
    write_contract_to_lakehouse,
    profile_for_dq,
    suggest_dq_rules,
    extract_dq_rules,
    review_dq_rules,
    build_dq_rule_history,
)


## Agreement and approved usage


In [ ]:
USE_SAMPLE_DATA = True
DATA_AGREEMENT = "sample_minimal_agreement" if USE_SAMPLE_DATA else "TODO_replace_agreement"
APPROVED_USAGE = "template_proof_path" if USE_SAMPLE_DATA else "TODO_replace_approved_usage"
ENV_NAME = "dev"
SOURCE_LAYER = "source"
SOURCE_KIND = "lakehouse"
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "TODO_source_table"
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "topic_dataset"
TARGET_TABLE = "sample_agreement_output" if USE_SAMPLE_DATA else f"{DATASET_NAME}_output"
DQ_TABLE_NAME = TARGET_TABLE


In [ ]:
CONFIG = load_fabric_config(CONFIG)
validate_notebook_name()
assert_notebook_name_valid()
runtime_context = build_runtime_context(dataset_name=DATASET_NAME, environment=ENV_NAME, source_table=SOURCE_TABLE, target_table=TARGET_TABLE)
if not USE_SAMPLE_DATA:
    bootstrap_fabric_env(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])


In [ ]:
source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG) if not USE_SAMPLE_DATA else None


In [ ]:
if SOURCE_KIND == "lakehouse":
    if USE_SAMPLE_DATA:
        source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
        seed_minimal_sample_source_table(source_path, table_name=SOURCE_TABLE)
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
print(df_source.schema)
display(df_source.limit(20))


## Segment 2: Profile source and capture evidence

What this does: loads source data and builds profiling metadata for review.

Functions used: `lakehouse_table_read`/`warehouse_read`, `generate_metadata_profile`, `profile_dataframe_to_metadata`.

Output: source profile rows for human review and downstream drafting.


In [ ]:
profile_rows = profile_for_dq(df_source, table_name=DQ_TABLE_NAME)
display(profile_rows)


## Exploration findings
Document EDA findings here.


## Transformation rationale
Document approved transformation rationale for pipeline handoff.


## Segment 3: AI-assisted DQ drafting (advisory only)

What this does: uses profile metadata plus configured prompts to generate advisory candidate DQ rules.

Functions used: `draft_dq_rules`, `review_dq_rules`, `write_dq_rules`.


In [ ]:
responses = suggest_dq_rules(
    profile_rows,
    prompt_template=CONFIG.ai_prompt_config.dq_rule_candidate_template,
    output_col='ai_response',
)
CANDIDATE_DQ_RULES = extract_dq_rules(responses, table_name=DQ_TABLE_NAME)
review_dq_rules(CANDIDATE_DQ_RULES, table_name=DQ_TABLE_NAME)


## Draft source input contract

This notebook helps humans define the **source input contract** for the pipeline.

- Capture required columns, optional columns, business keys, classifications, and approved DQ rules.
- In Fabric-first usage, draft this as a Python dict or small table, then write approved values to metadata tables.
- AI suggestions are advisory only and require human/steward approval.
- Optional export formats can be added in project implementations.


In [ ]:
# Review widget writes explicit approvals to dq_review.APPROVED_RULES_FROM_WIDGET.
HUMAN_APPROVED_RULES = list(dq_review.APPROVED_RULES_FROM_WIDGET)
if not HUMAN_APPROVED_RULES:
    raise ValueError("No approved DQ rules selected in widget. Approve at least one rule before persisting.")

approved_rules_metadata_df = build_dq_rule_history(
    spark=spark,
    table_name=DQ_TABLE_NAME,
    approved_rules=HUMAN_APPROVED_RULES,
    action_by='notebook_user',
)
display(approved_rules_metadata_df)

approved_rules_metadata_df.write.mode('append').saveAsTable('METADATA_DQ_RULES')
print("Stored approved DQ rules in METADATA_DQ_RULES")


## Human-reviewed DQ decisions
Copy only approved deterministic rules to 03_pc notebook.


In [ ]:
SOURCE_INPUT_CONTRACT_DRAFT = {
    "contract_id": f"{DATA_AGREEMENT}:{SOURCE_TABLE}:v1",
    "contract_type": "source_input",
    "dataset_name": DATASET_NAME,
    "object_name": SOURCE_TABLE,
    "version": "1.0.0",
    "status": "approved",
    "required_columns": ["customer_id", "event_ts", "status", "amount"],
    "optional_columns": ["email", "country_code"],
    "business_keys": ["customer_id", "event_ts"],
    "classifications": {
        "customer_id": "internal",
        "event_ts": "internal",
        "status": "internal",
        "amount": "internal",
        "email": "confidential",
        "country_code": "internal",
    },
    "quality_rules": HUMAN_APPROVED_RULES,
    "column_types": {
        "customer_id": "string",
        "event_ts": "timestamp",
        "status": "string",
        "amount": "double",
        "email": "string",
        "country_code": "string",
    },
    "approved_by": "notebook_user",
    "approval_note": "Reviewed in 02_ex; approved for deterministic 03_pc enforcement.",
}

SOURCE_INPUT_CONTRACT_APPROVED = normalize_contract_dict(SOURCE_INPUT_CONTRACT_DRAFT)
display(SOURCE_INPUT_CONTRACT_APPROVED)


## Segment 4: Human approval and contract write

What this does: captures approved classifications/rules and writes approved contract metadata.

Functions used: `classify_columns`, `summarize_governance_classifications`, `validate_contract_dict`, `write_contract_to_lakehouse`.

Output: approved contract and governance evidence for pipeline enforcement.


## Human-reviewed classification decisions
Approve labels with governance stewards.


In [ ]:
# Optional governance evidence omitted for minimal path.


## Approve and write source input contract
Validate draft contracts, then explicitly gate writes to the dedicated metadata target.


In [ ]:
contract_errors = validate_contract_dict(SOURCE_INPUT_CONTRACT_APPROVED)
if contract_errors:
    raise ValueError("Contract validation failed: " + " | ".join(contract_errors))

WRITE_APPROVED_CONTRACT = True
USE_LOCAL_SAMPLE_METADATA = False
# Set True only when running locally without Fabric metadata tables.

if WRITE_APPROVED_CONTRACT and not USE_LOCAL_SAMPLE_METADATA:
    metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)
    write_contract_to_lakehouse(SOURCE_INPUT_CONTRACT_APPROVED, metadata_path)
elif USE_LOCAL_SAMPLE_METADATA:
    out = Path("samples/end_to_end/_output/metadata")
    out.mkdir(parents=True, exist_ok=True)
    (out / "contracts.json").write_text(json.dumps([SOURCE_INPUT_CONTRACT_APPROVED], default=str, indent=2), encoding="utf-8")
    print("Wrote local sample metadata artifact: samples/end_to_end/_output/metadata/contracts.json")


In [ ]:
from fabricops_kit.data_lineage import build_lineage_from_notebook_code

# Optional documentation helper only (not transformation logic).
# Paste or load exported notebook source when ready to document lineage.
# result = build_lineage_from_notebook_code(notebook_code)
# display(result["deterministic_steps"])
# print(result["copilot_prompt"])
